# Semaine 9 — Jour 3 : Vector store, persistance et filtres

Persister les passages, vecteurs et métadonnées dans un store local.

Parcours essentiel : 60 minutes (cours 20 min, lab 25 min, exercice 10 min, synthèse 5 min). Prévoir 2 à 3 heures pour l’unité complète avec entretien et challenge. Les modèles et services optionnels ajoutent un temps d’installation distinct.

Prérequis : Python 3.11+, JSON, notions de vecteurs et de persistance. Le parcours obligatoire utilise uniquement la bibliothèque standard et ne contacte aucun service externe.

Lire [les objectifs](learning_objectives.md), [le chapitre](chapter.md), puis [le lab](labs/README.md). Réaliser [les exercices](exercises.md), [l’entretien](interview.md) et [le challenge](challenge.md) avant les corrigés locaux.

Notebooks depuis la racine : `notebooks/week09/S9_J3_vector_store.ipynb` et `notebooks/week09/teacher/S9_J3_vector_store_teacher.ipynb`.

Critère de réussite : Mettre à jour ou supprimer un document sans conserver ses anciennes projections.

Les calculs de retrieval et de persistance sont exécutables; le corpus est fictif. L’encodeur de base est lexical et la réponse finale est extractive. Les extensions neuronales sont séparées et n’ont pas été exécutées ici.


# Objectifs mesurables

- Persister les passages, vecteurs et métadonnées dans un store local.
- Appliquer le périmètre avant le top-k et vérifier la persistance après réouverture.
- Mettre à jour ou supprimer un document sans conserver ses anciennes projections.

## Preuves attendues

Une sortie reproductible, un contrôle d’invariant et une limite du mécanisme correctement expliquée.


# S9 J3 — Vector store, persistance et filtres


## 1. Un vecteur seul ne suffit pas
Un système documentaire doit stocker texte, source, version et droits avec la représentation vectorielle. Une liste de nombres sans provenance ne permet ni citation ni suppression cohérente. Un vector store définit aussi une stratégie d’ingestion, de filtrage, d’évolution et de récupération après panne.

Le parcours obligatoire utilise SQLite pour la persistance et Python pour un scan exact des vecteurs autorisés. SQLite n’est pas configuré comme une base vectorielle avec index ANN. Ce choix rend la mécanique visible et testable hors ligne sur quelques dizaines de passages. L’extension pgvector est étudiée dans un lab optionnel séparé, sans prétendre avoir été exécutée.

## 2. Contrat de la collection
La table documents garde le document courant par couple tenant/doc_id. La table chunks garde les projections et leurs vecteurs. La table metadata conserve l’empreinte de l’encodeur. La même version documentaire avec un autre contenu est refusée; une version inférieure à la courante l’est aussi.

L’ingestion prépare chunks et vecteurs, puis supprime les anciennes projections et écrit les nouvelles dans une transaction. Refaire le même contenu peut reconstruire le découpage; cela ne crée pas de doublons. Un échec d’encodage arrive avant la mutation. Un échec SQL dans la transaction doit laisser l’ancienne projection exploitable.

## 3. Filtrer avant de classer
scoped_rows exécute une requête SQL WHERE tenant=? avant tout calcul de score. La sélection top-k n’utilise donc que le périmètre voulu. Un document team-b peut contenir exactement la requête et rester absent des résultats team-a. Filtrer après un top-k global peut éliminer tous les candidats utiles et gaspiller la recherche sur un autre périmètre.

Le paramètre tenant suppose une décision d’autorisation déjà prise côté service. Le CLI de laboratoire permet de choisir le tenant pour tester les cas; il ne fournit pas d’authentification. Un service réel dériverait le périmètre de l’identité et des ressources autorisées, avec éventuellement des droits plus fins que le seul tenant.

## 4. Recherche exacte et ANN
Le scan exact calcule toutes les similarités dans le périmètre, puis trie. Il sert de baseline. Un index approximatif réduit souvent le coût de recherche mais introduit un compromis entre rappel, latence, mémoire et mise à jour. Le mesurer sur des requêtes représentatives plutôt que promettre le même résultat à partir du seul nom HNSW.

Avec des filtres et un index ANN, la mécanique d’exécution demande une attention particulière : le nombre de candidats réellement éligibles peut être plus petit que prévu. Le dossier optionnel présente pgvector, son opérateur de distance cosinus et une requête paramétrée. L’étude des scans itératifs et des paramètres d’index doit être faite sur l’installation réelle.

## 5. Cycle de vie
Une nouvelle version remplace les chunks de la précédente. Une citation contenant un ancien chunk_id devient invalide. delete_document supprime document, chunks et mémoires référencées. Cette opération ne garantit pas l’effacement des anciennes sauvegardes ou copies externes; elle définit le comportement observable du store local.

Le modèle est une autre dimension de version. Changer son empreinte exige un index distinct et une reconstruction. Une migration propre peut conserver ancien et nouvel index le temps de comparer les résultats puis basculer un pointeur de service. Le lab ne fournit pas cet opérateur de bascule ni un cluster distribué.

## 6. Preuve du jour
Écrire dans une base fichier, fermer puis rouvrir. Vérifier les mêmes identifiants classés. Ajouter une version 2 et vérifier l’absence des anciens passages. Faire correspondre la suppression à l’absence de résultats et à l’invalidation d’une citation. Ces preuves sont plus utiles que la seule réussite d’un INSERT.

## À retenir
La qualité d’un store vectoriel dépend autant de ses filtres, versions et suppressions que du calcul de distance. Le moteur local prouve ces contrats sur un processus, pas une capacité distribuée.


## Diagramme du jour

```mermaid
flowchart TD
 A["Document courant"] --> B["Encodage validé"]
 B --> C["Transaction de remplacement"]
 C --> D["SQLite persistant"]
 E["Tenant autorisé"] --> F["Filtre SQL"]
 D --> F
 F --> G["Scan exact puis top-k"]
```

Le rendu Mermaid dépend du lecteur; la source .mmd reste disponible.

# Lab exécutable

Les cellules s’exécutent hors ligne depuis l’arbre du dépôt.

In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / "examples/week09/knowledge.py").is_file()), None)
assert root is not None, "Ouvrir le notebook depuis le dépôt décompressé"
sys.path.insert(0, str(root))
import tempfile
from pathlib import Path
from examples.week09.demo import populated
from examples.week09.knowledge import Store
with tempfile.TemporaryDirectory() as directory:
    path = str(Path(directory) / "index.sqlite")
    store = populated(path)
    before = store.search("paiement Kafka", "team-a", top_k=3)
    store.close()
    reopened = Store(path)
    try:
        after = reopened.search("paiement Kafka", "team-a", top_k=3)
        assert [c["chunk_id"] for c in before] == [c["chunk_id"] for c in after]
        assert all(c["tenant"] == "team-a" for c in after)
    finally:
        reopened.close()
print("Persistance et filtre vérifiés")

# Lab J3 — Vector store, persistance et filtres

Depuis la racine du dépôt :

```bash
python3 book/week09/day03/labs/run.py
python3 -m examples.week09.demo --day 3
```

Résultat attendu : **persistent=true, cross_tenant=false**.

## Manipulations

1. Reproduire la sortie et retrouver le calcul dans `examples/week09/knowledge.py`.
2. Réaliser l’exercice 2 avec une copie des données, sans modifier le corpus distribué.
3. Vérifier l’invariant étudié et expliquer une limite.

Le corpus se trouve dans `examples/week09/corpus`. Les tests automatisés sont sous `tests/week09`. [QUICKSTART](../../../../docs/week09/QUICKSTART.md) décrit les commandes et les codes retour.

Les modèles locaux et pgvector sont optionnels : lire `examples/week09/optional/README.md` depuis la racine du dépôt. Ils ne sont pas nécessaires pour finir cette journée et ne sont pas présentés comme validés dans cet environnement.


# Exercices

Conserver votre première réponse avant de lire les solutions.

## Exercice 1 — Relier les tables

Pourquoi stocker documents et chunks séparément ?

Livrable : réponse argumentée avec calcul, test ou observation.

## Exercice 2 — Choisir le filtre

Expliquer le défaut d’un top-k global suivi d’un filtrage de tenant.

Livrable : réponse argumentée avec calcul, test ou observation.

## Exercice 3 — Tester un remplacement

Quels objets deviennent invalides après une mise à jour documentaire ?

Livrable : réponse argumentée avec calcul, test ou observation.


# Entretien

Répondre en deux minutes avec un exemple concret et une limite.

## Question 1

Pourquoi commencer par une recherche exacte ?

## Question 2

Un WHERE tenant est-il une authentification ?


# Challenge

Indexer le corpus, conserver une citation puis modifier son document avec version=2. Vérifier que la citation ancienne est refusée. Fermer et rouvrir la base, puis supprimer le document.

## Critères de réussite

Reproductibilité, provenance des données, invariant vérifié et limite expliquée sans exagérer les garanties.


# Références — J3

Sources primaires consultées le 20 octobre 2026. Les algorithmes et contrats du lab sont explicités dans le chapitre; le corpus est une création fictive du cours.

- [pgvector : documentation officielle](https://github.com/pgvector/pgvector) — stockage, distances, recherche exacte/ANN et filtres.
- [Sentence Transformers : retrieve and rerank](https://www.sbert.net/examples/sentence_transformer/applications/retrieve_rerank/README.html) — architecture de récupération en deux étapes.
- [SentenceTransformer : API](https://www.sbert.net/docs/package_reference/sentence_transformer/model.html) — encode_query, encode_document et modèles locaux.
- [CrossEncoder : API](https://www.sbert.net/docs/package_reference/cross_encoder/model.html) — scoring conjoint de paires.
- [Cormack, Clarke et Buettcher, SIGIR 2009 : Reciprocal Rank Fusion](https://research.google/pubs/reciprocal-rank-fusion-outperforms-condorcet-and-individual-rank-learning-methods/) — origine de la méthode de fusion des rangs.
- [Python sqlite3](https://docs.python.org/3/library/sqlite3.html) — persistance locale et transactions.

Lire en priorité les références du thème du jour. Les extensions évoluent; leur environnement réel doit être verrouillé et testé avant utilisation. Le paquet n’affirme pas avoir exécuté de modèle neuronal ou de serveur pgvector.
